In [1]:
import argparse
import json
import os
import tempfile
import numpy as np
import subprocess
from tqdm import tqdm
import pandas as pd
import datetime
import time
import gc
import random
import sys
import shutil
import glob

sys.path.append('/home/agustin/phd/synthesis')
# sys.path.append('/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/test4_finetune_brainst_diffusion_model/training/networks_declaration')


# pytorch
import torch
from torch.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, Sampler
import torch.distributed as dist
# data loader
from sklearn.preprocessing import MinMaxScaler

# mine
import utils.nifti_functions as nfc
import utils.util as util
import utils.functions as fc
import utils.util_freesurfer_segmentation as ufs
import utils.gpu_selector as gpu_selector
import data_loaders.load_dataset as load_dataset
import utils.data_normalization as data_normalization


# monai
from monai.bundle import ConfigParser

import  networks_declaration.diffusion_model_unet_maisi_mask_att as diffusion_model_unet_maisi
import  networks_declaration.volumne_encoder as volumne_encoder


# import attention_controller as attention_controller
from networks_declaration.rectified_flow import RFlowScheduler
# from monai.networks.schedulers.rectified_flow import RFlowScheduler
from monai.networks.schedulers.ddpm import DDPMPredictionType
# images
from PIL import Image

sys.path.append('/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/utils')
from autoencoder_declaration import AutoencoderPrediction

# device_name = f"cuda:{gpu_selector.get_least_used_gpu()}"
device_name = f"cuda:1"
device = torch.device(device_name)


def set_seed(seed: int):
    # random.seed(seed)  # Semilla para Python
    np.random.seed(seed)  # Semilla para NumPy
    torch.manual_seed(seed)  # Semilla para PyTorch en CPU
    torch.cuda.manual_seed(seed)  # Semilla para PyTorch en GPU
    torch.cuda.manual_seed_all(seed)  # Semilla para todas las GPUs
    torch.backends.cudnn.deterministic = True  # Garantizar reproducibilidad en CNNs
    torch.backends.cudnn.benchmark = False  # Desactivar optimización no determinista


In [2]:
import  networks_declaration.segmentation_encoder as segmentation_encoder


segmentation_encoder_def = {
    "spatial_dims": 3,
    "in_channels": 4,
    "num_res_blocks": 2,
    "num_channels": [
        32,
        64,
        128,
        256,
    ]
}

segmentation_encoder_model = segmentation_encoder.SegmentationEncoder(
    spatial_dims=segmentation_encoder_def["spatial_dims"],
    in_channels=segmentation_encoder_def["in_channels"],
    num_res_blocks=segmentation_encoder_def["num_res_blocks"],
    num_channels=segmentation_encoder_def["num_channels"],
)

In [6]:
# import torch
# from monai.networks.blocks import ResidualUnit, Convolution
# import torch.nn as nn

# class MaskEncoder(nn.Module):
#     def __init__(
#         self,
#         spatial_dims,
#         in_channels,
#         num_channels,
#         num_res_blocks=2,
#     ):
#         super().__init__()

#         assert spatial_dims == 3, "Only 3D supported for now"

#         self.num_levels = len(num_channels)

#         # stem (match first UNet channel)
#         # self.stem = nn.Conv3d(in_channels, num_channels[0], kernel_size=1)

#         self.stem = Convolution(
#             spatial_dims=3,
#             in_channels=in_channels,
#             out_channels=num_channels[0],
#             strides=1,
#             kernel_size=1,
#             act="PRELU",
#             norm="INSTANCE",
#         )

#         # encoder layers
#         self.blocks = nn.ModuleList()
#         self.downsamples = nn.ModuleList()

#         for i in range(self.num_levels):

#             ch = num_channels[i]

#             # residual blocks per level
#             res_blocks = ResidualUnit(
#                     spatial_dims=3,
#                     in_channels=ch,
#                     out_channels=ch,
#                     subunits=num_res_blocks,
#                     act="PRELU",
#                     norm="INSTANCE",
#                 )
            
#             self.blocks.append(res_blocks)

#             # downsample (except last level)
#             if i != self.num_levels - 1:
#                 self.downsamples.append(
#                     Convolution(
#                         spatial_dims=3,
#                         in_channels=ch,
#                         out_channels=num_channels[i + 1],
#                         strides=2,
#                         kernel_size=3,
#                         act="PRELU",
#                         norm="INSTANCE",
#                     )
#                 )

#     def forward(self, x):

#         features = []

#         x = self.stem(x)

#         for i in range(self.num_levels):

#             x = self.blocks[i](x)
#             features.append(x)

#             if i < self.num_levels - 1:
#                 x = self.downsamples[i](x)

#         return {
#             f"f{i}": features[i] for i in range(len(features))
#         }
    

# mask_encoder_config = {
#     "spatial_dims": 3,
#     "in_channels": 4,
#     "num_res_blocks": 2,
#     "num_channels": [
#         32,
#         64,
#         128,
#         # 256,
#     ]
# }

# mask_encoder = MaskEncoder(**mask_encoder_config).to(device)
# print("Mask encoder parameters:", sum(p.numel() for p in mask_encoder.parameters()))
# # print(mask_encoder)




# # Number of parameters in MaskEncoder: 1438505



In [7]:
# from torch import nn

# from monai.networks.blocks import ResidualUnit, Convolution

# class MaskEncoder(nn.Module):
#     def __init__(self, in_channels, base_ch=32):
#         super().__init__()

#         self.stem = Convolution(
#             spatial_dims=3,
#             in_channels=in_channels,
#             out_channels=base_ch,
#             strides=1,
#             kernel_size=1,
#             act="PRELU",
#             norm="INSTANCE",
#         )

#         self.enc1 = ResidualUnit(3, base_ch, base_ch)
#         self.down1 = Convolution(3, base_ch, base_ch*2, strides=2)

#         self.enc2 = ResidualUnit(3, base_ch*2, base_ch*2)
#         self.down2 = Convolution(3, base_ch*2, base_ch*4, strides=2)

#         self.enc3 = ResidualUnit(3, base_ch*4, base_ch*4)

#     def forward(self, x):
#         x0 = self.stem(x)
#         x1 = self.enc1(x0)
#         x2 = self.enc2(self.down1(x1))
#         x3 = self.enc3(self.down2(x2))

#         return {
#             "f0": x1,
#             "f1": x2,
#             "f2": x3
#         }
    
# mask_encoder = MaskEncoder(in_channels=4, base_ch=32)
# nb_parameters = sum(p.numel() for p in mask_encoder.parameters())
# print(f"Number of parameters in MaskEncoder: {nb_parameters}")
# mask_encoder

### UNET

In [15]:
import  networks_declaration.diffusion_model_unet_maisi_mask_seg as diffusion_model_unet_maisi

def instantiate_unconditioned_models():

    networks_config =  {
        
        "diffusion_unet_def": {
            "_target_": "monai.apps.generation.maisi.networks.diffusion_model_unet_maisi.DiffusionModelUNetMaisi",
            "spatial_dims": 3,
            "in_channels": 4,
            "out_channels": 4,
            "num_res_blocks": 2,
            "num_channels": [
                64,
                128,
                256,
                512
            ],
            "self_attention_levels": [
                False,
                False,
                False,
                False
            ],

            "num_self_head_channels": [
                0,
                0,
                0,
                1
            ],
            "cross_attention_levels": [
                False,
                False,
                False,
                False
            ],
            "num_cross_head_channels": [
                0,
                0,
                0,
                0
            ],

            "use_flash_attention": True,
            "with_conditioning": False,
            "cross_attention_dim": None,
            "transformer_num_layers": 1, # number of transformer blocks
            "upcast_attention": True,

        },


    }


    # instantiate model
    parser = ConfigParser(networks_config)
    parser.parse(True)

    args = fc.dict_to_args(networks_config, deep_conversion=True)

    # unet
    unet = diffusion_model_unet_maisi.DiffusionModelUNetMaisi(spatial_dims = args.diffusion_unet_def.spatial_dims,
                                                in_channels = args.diffusion_unet_def.in_channels,
                                                out_channels = args.diffusion_unet_def.out_channels,
                                                num_res_blocks = args.diffusion_unet_def.num_res_blocks,
                                                num_channels = args.diffusion_unet_def.num_channels,
                                                # attention_levels = args.diffusion_unet_def.attention_levels,
                                                self_attention_levels = args.diffusion_unet_def.self_attention_levels,
                                                cross_attention_levels = args.diffusion_unet_def.cross_attention_levels,
                                                # num_head_channels = args.diffusion_unet_def.num_head_channels,
                                                num_self_head_channels = args.diffusion_unet_def.num_self_head_channels,
                                                num_cross_head_channels = args.diffusion_unet_def.num_cross_head_channels,
                                                with_conditioning = args.diffusion_unet_def.with_conditioning,
                                                transformer_num_layers = args.diffusion_unet_def.transformer_num_layers,
                                                cross_attention_dim = args.diffusion_unet_def.cross_attention_dim,
                                                upcast_attention = args.diffusion_unet_def.upcast_attention,
                                                use_flash_attention = args.diffusion_unet_def.use_flash_attention,
                                                include_top_region_index_input=False,
                                                include_bottom_region_index_input=False,
                                                include_spacing_input=False
                                                )
    

    return {"unet": unet, 
              "networks_config": args}



models_dict = instantiate_unconditioned_models()
unet = models_dict["unet"]


unet.to(device)
unet.train()
a =2
# unet
print("number of parameters in unet: ", sum(p.numel() for p in unet.parameters()))
print(unet)

number of parameters in unet:  160025348
DiffusionModelUNetMaisi(
  (conv_in): Convolution(
    (conv): Conv3d(4, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (time_embed): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): SiLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock(
      (resnets): ModuleList(
        (0-1): 2 x DiffusionUNetResnetBlock(
          (norm1): GroupNorm(32, 64, eps=1e-06, affine=True)
          (nonlinearity): SiLU()
          (conv1): Convolution(
            (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (time_emb_proj): Linear(in_features=256, out_features=64, bias=True)
          (norm2): GroupNorm(32, 64, eps=1e-06, affine=True)
          (conv2): Convolution(
            (conv): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          )
          (s